# End-to-End Pipeline Demo — M4Runs the complete pricing pipeline:  **Preprocess → Cluster → Regress → Price**  on the full dataset and shows per-customer results.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib
matplotlib.use("Agg")

from src.pipeline import run_full_pipeline

print("Imports OK | project root:", PROJECT_ROOT)

Imports OK | project root: /home/taqis/Pone/CSCI323_Project


In [2]:
result = run_full_pipeline(
    str(PROJECT_ROOT / "data" / "raw" / "medical_insurance.csv"),
    save_results=True,
)
print("Pipeline complete.")
print("Keys:", list(result.keys()))

Pipeline complete.
Keys: ['data', 'clustering', 'regression', 'pricing_results']


In [3]:
# ── Clustering summary ──────────────────────────────────────────────────────
c = result["clustering"]
print(f"K-Means k={c['k']}, silhouette={c['silhouette_score']:.4f}")

from collections import Counter
for tier, n in sorted(Counter(c["risk_tier"]).items()):
    print(f"  {tier:<14}: {n} training customers")

K-Means k=2, silhouette=0.3693
  HIGH_RISK     : 517 training customers
  LOW_RISK      : 553 training customers


In [4]:
# ── Regression summary ──────────────────────────────────────────────────────
r = result["regression"]
print(f"Best model : {r['best_model_name']}")
print()
print("Validation metrics:")
for k, v in r["val_metrics"].items():
    print(f"  {k:<6}: {v:,.4f}")
print()
r["comparison_table"]

Best model : Linear Regression

Validation metrics:
  rmse  : 5,572.6678
  mae   : 4,042.0166
  r2    : 0.7822



,Model,Train RMSE,Train MAE,Train R²,Val RMSE,Val MAE,Val R²
0,Linear Regression,6105.245135,4207.976585,0.741751,5572.667818,4042.016644,0.782227
1,Ridge,6105.493012,4217.235290,0.741730,5580.997516,4057.014503,0.781575
2,Lasso,6105.247735,4208.046260,0.741751,5573.360816,4042.785959,0.782173


In [5]:
# ── Pricing results ─────────────────────────────────────────────────────────
pricing = result["pricing_results"]
print(f"Customers priced: {len(pricing)}")
print()
print("Mean premium by risk tier:")
print(pricing.groupby("risk_tier")["recommended_premium"].mean().round(2).to_string())
print()
print("Discount breakdown:")
eligible = (pricing["discount_applied"] > 0).sum()
print(f"  Discount applied : {eligible} ({eligible/len(pricing)*100:.1f}%)")
print()
pricing.head(10)

Customers priced: 134

Mean premium by risk tier:
risk_tier
HIGH_RISK    2152.51
LOW_RISK      773.55

Discount breakdown:
  Discount applied : 32 (23.9%)



,predicted_expense,risk_tier,base_premium,risk_multiplier,risk_adjusted_premium,discount_applied,recommended_premium
0,3930.20,LOW_RISK,393.02,0.85,334.07,0.00,334.07
1,26857.22,LOW_RISK,2685.72,0.85,2282.86,228.29,2054.58
2,2370.69,LOW_RISK,237.07,0.85,201.51,0.00,201.51
3,9357.30,LOW_RISK,935.73,0.85,795.37,0.00,795.37
4,31809.85,LOW_RISK,3180.98,0.85,2703.84,270.38,2433.45
5,33461.40,HIGH_RISK,3346.14,1.25,4182.68,418.27,3764.41
6,7226.69,HIGH_RISK,722.67,1.25,903.34,0.00,903.34
7,31796.08,HIGH_RISK,3179.61,1.25,3974.51,397.45,3577.06
8,9116.85,LOW_RISK,911.69,0.85,774.93,0.00,774.93
9,10928.59,HIGH_RISK,1092.86,1.25,1366.07,0.00,1366.07


In [6]:
# ── Premium distribution plot ────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, tier, color in zip(
    axes,
    ["LOW_RISK", "HIGH_RISK"],
    ["steelblue", "tomato"],
):
    subset = pricing[pricing["risk_tier"] == tier]["recommended_premium"]
    ax.hist(subset, bins=20, color=color, alpha=0.8, edgecolor="white")
    ax.set_title(f"{tier} — Premium Distribution")
    ax.set_xlabel("Recommended Premium ($)")
    ax.set_ylabel("Count")
    ax.axvline(subset.mean(), color="black", linestyle="--", label=f"Mean: ${subset.mean():,.0f}")
    ax.legend()

plt.tight_layout()
out = PROJECT_ROOT / "outputs" / "figures"
out.mkdir(parents=True, exist_ok=True)
plt.savefig(out / "premium_distribution_by_tier.png", dpi=150)
plt.show()
print("Plot saved.")

Plot saved.


/tmp/ipykernel_426145/895721828.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ── Contract check ──────────────────────────────────────────────────────────
assert {"data", "clustering", "regression", "pricing_results"} == set(result.keys())
assert len(pricing) > 0
assert (pricing["recommended_premium"] >= 0).all()
assert pricing["risk_tier"].isin(["LOW_RISK", "MEDIUM_RISK", "HIGH_RISK"]).all()

required_cols = {
    "predicted_expense", "risk_tier", "base_premium",
    "risk_multiplier", "risk_adjusted_premium",
    "discount_applied", "recommended_premium",
}
assert required_cols.issubset(set(pricing.columns))

print("All M4 contract checks passed.")

All M4 contract checks passed.
